**My Personal Finance Buddy**

**Step 1 : Data collection & Store embeddings in ChromaDB**
- Collect transcripts from YouTube: download audio via yt-dlp, then transcribe with Whisper
- Use task="transcribe" when lang="en"
- Use task="translate" otherwise (e.g., FR → EN)
- Preprocess & chunk with RecursiveCharacterTextSplitter into ~500 characters with 100 overlap
- Embed & store the chunks in ChromaDB using OpenAIEmbeddings

In [1]:
# ----------------- Cell 1: YouTube Processing -----------------
import os
import re
import subprocess
import sys
from pathlib import Path

import whisper
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

video_map = {
    "ouvbeb2wSGA": {"name": "General_Literacy", "lang": "en"},
    "bb6_M_srMBk": {"name": "Investment", "lang": "en"},
    "MabD5R8kRak": {"name": "Financial_Goal", "lang": "en"},
    "7XVmhedNiIQ": {"name": "Changing_Habits", "lang": "en"},
    "pyEgYI7S-r4": {"name": "Credit_and_Debts", "lang": "en"},
    "q6d4VAIWFPA": {"name": "Level_of_Financial_Liberty", "lang": "fr"},
}

# Works on macOS, Windows and Linux. Override with FINANCE_BUDDY_DIR in .env if needed.
BASE_DIR = Path(os.getenv("FINANCE_BUDDY_DIR", Path.cwd() / "data"))
OUTPUT_DIR = BASE_DIR / "YT_video"
CHROMA_DB_PATH = str(BASE_DIR / "chroma_db")

_whisper_models = {}

def load_whisper(model_size):
    # Load each Whisper model once instead of once per video
    if model_size not in _whisper_models:
        _whisper_models[model_size] = whisper.load_model(model_size)
    return _whisper_models[model_size]

def download_and_transcribe(video_id, model_size="base"):
    video_info = video_map.get(video_id, {"name": video_id, "lang": "en"})
    name, lang = video_info["name"], video_info["lang"]

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    audio_path = OUTPUT_DIR / f"{name}.mp3"
    transcript_path = OUTPUT_DIR / f"{name}.txt"

    # Use cached transcript if exists
    if transcript_path.exists():
        return transcript_path.read_text(encoding="utf-8")

    # Download audio if not already there
    if not audio_path.exists():
        subprocess.run(
            [sys.executable, "-m", "yt_dlp", "-x", "--audio-format", "mp3",
             f"https://www.youtube.com/watch?v={video_id}",
             "-o", str(OUTPUT_DIR / f"{name}.%(ext)s")],
            check=False,
        )

    if not audio_path.exists():
        raise FileNotFoundError(f"❌ Failed: {audio_path} not created.")

    # Whisper transcription (translate FR → EN if needed)
    model = load_whisper(model_size)
    task = "transcribe" if lang == "en" else "translate"
    result = model.transcribe(str(audio_path), task=task, language=lang)

    transcript_path.write_text(result["text"], encoding="utf-8")
    return result["text"]

def preprocess_and_chunk(transcripts, chunk_size=500, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    docs = []
    for video_id, text in transcripts.items():
        cleaned = re.sub(r"\s+", " ", text).strip()
        topic = video_map.get(video_id, {}).get("name", video_id)
        doc = Document(page_content=cleaned, metadata={"source": video_id, "topic": topic})
        docs.extend(splitter.split_documents([doc]))
    return docs

def create_or_load_vector_db(documents=None):
    embeddings = OpenAIEmbeddings()
    if documents:
        # langchain_chroma persists automatically, no .persist() needed
        return Chroma.from_documents(documents, embeddings, persist_directory=CHROMA_DB_PATH)
    if os.path.exists(CHROMA_DB_PATH):
        return Chroma(persist_directory=CHROMA_DB_PATH, embedding_function=embeddings)
    return None

def build_knowledge_base(model_size="base"):
    transcripts = {}
    for video_id in video_map:
        try:
            transcripts[video_id] = download_and_transcribe(video_id, model_size)
        except Exception as e:
            print(f"⚠️ Skipping {video_id}: {e}")
    if not transcripts:
        raise RuntimeError("❌ No transcript could be created: check that yt-dlp and ffmpeg are installed.")
    docs = preprocess_and_chunk(transcripts)
    print(f"✅ {len(transcripts)} transcripts → {len(docs)} chunks stored in {CHROMA_DB_PATH}")
    return create_or_load_vector_db(docs)

# Build the knowledge base once; afterwards the persisted DB is reused.
# (Delete the chroma_db folder to rebuild it.)
vector_store = create_or_load_vector_db()
if vector_store is None or not vector_store.get(limit=1)["ids"]:
    # No DB yet, or an empty one left by a failed run (e.g. OpenAI quota error)
    vector_store = build_knowledge_base()
else:
    print(f"✅ Loaded knowledge base: {len(vector_store.get()['ids'])} chunks")


✅ Loaded knowledge base: 489 chunks


**Step 2 : RAG (Vector Store + create_rag_chain)**
- Setup: reuse the persisted Chroma vector store from Step 1; initialize GPT-4o (low temp).

- Retrieval: similarity search returning the top 3 transcript chunks.

- QA Chain: History-aware retriever (rewrites follow-up questions into standalone ones) + strict context-only prompt → answer generation via create_stuff_documents_chain

In [4]:
# ----------------- Cell 2: RAG Setup -----------------
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

def create_rag_chain(llm=None, vector_store=None):
    if vector_store is None:
        embeddings = OpenAIEmbeddings()
        vector_store = Chroma(persist_directory=CHROMA_DB_PATH, embedding_function=embeddings)

    if not llm:
        llm = ChatOpenAI(model="gpt-4o", temperature=0.2)

    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    contextualize_prompt = ChatPromptTemplate.from_messages([
        ("system", "Given the conversation history and the latest user question, rewrite the question "
                   "so it can be understood without the history. Do NOT answer it, only return the question."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ])
    history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_prompt)

    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are 'Coach'. Use ONLY the following context to answer. "
                   "If the context does not contain the answer, say that your sources don't cover it.\n\n{context}"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ])
    doc_chain = create_stuff_documents_chain(llm, qa_prompt)

    return create_retrieval_chain(history_aware_retriever, doc_chain)

rag_chain = create_rag_chain(vector_store=vector_store)


**Step 3 : Agent (Tools + Flow)**
- `knowledge_base` tool: answers financial-literacy questions from the YouTube transcripts (RAG from Step 2).
- `create_plan` tool: collects the user's financial profile (house, savings, trip). The LLM extracts every detail the user gives, in any order; the tool asks for the next missing one.
- Evaluates financial health (Strong/Moderate/Weak) from savings rate, debt / annual income and emergency-fund months, and gives advice.
- Enriches plans with live web data (e.g., mortgage rates).
- Each user has their own session (profile + plans), so profiles are never shared between Gradio visitors.

In [5]:
# ----------------- Cell 3: Agent Setup -----------------
import contextvars
import re
from datetime import date
from typing import Optional

from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

# --- Per-user session state ---
# Each Gradio visitor gets their own session dict (see Cell 5); tools read it via get_session().
def new_session():
    return {"user_profile": {}, "plan_states": {}, "last_plan": None}

_current_session = contextvars.ContextVar("finance_session")

def use_session(session):
    _current_session.set(session)
    return session

def get_session():
    try:
        return _current_session.get()
    except LookupError:
        return use_session(new_session())

# --- Financial health ---
def financial_metrics(state):
    """income/expenses are monthly; savings/debt are current totals."""
    income = float(state.get("income") or 0)
    expenses = float(state.get("expenses") or 0)
    savings = float(state.get("savings") or 0)
    debt = float(state.get("debt") or 0)
    annual_income = income * 12
    return {
        # share of monthly income left after expenses
        "Savings rate": (income - expenses) / income if income else 0.0,
        "Debt / annual income": debt / annual_income if annual_income else (1.0 if debt else 0.0),
        "Emergency fund (months)": savings / max(expenses, 1),
    }

def evaluate_financial_health(state):
    try:
        m = financial_metrics(state)
    except (TypeError, ValueError):
        return "⚠️ Could not evaluate financial health.", "Unknown"

    savings_rate = m["Savings rate"]
    debt_ratio = m["Debt / annual income"]
    emergency_months = m["Emergency fund (months)"]

    reasons, consequences = [], []
    if savings_rate < 0.2:
        reasons.append("Low savings rate (<20% of income left each month)")
        consequences.append("Harder to build an emergency fund or down payment.")
    if debt_ratio >= 0.35:
        reasons.append("High debt load (≥35% of annual income)")
        consequences.append("Limits borrowing capacity and increases financial risk.")
    if emergency_months < 3:
        reasons.append("Emergency fund below 3 months of expenses")
        consequences.append("An unexpected expense could push you into debt.")

    if savings_rate >= 0.2 and debt_ratio < 0.2 and emergency_months >= 6:
        return (
            "💎 Strong – Excellent! You are financially strong. Focus on investments and long-term goals.",
            "Strong",
        )
    elif savings_rate >= 0.1 and debt_ratio < 0.5:
        return (
            f"📈 Moderate – On the right track. Issues: {', '.join(reasons) or 'None'}. "
            "Tip: boost savings rate for resilience.",
            "Moderate",
        )
    else:
        return (
            f"⚠️ Weak – Vulnerable finances. Issues: {', '.join(reasons) or 'General instability'}. "
            f"Consequences: {', '.join(consequences) or 'Difficulty reaching financial goals.'}",
            "Weak",
        )

# --- Plan schemas ---
PLAN_SCHEMAS = {
    "house": {
        "required": ["price", "savings", "income", "expenses", "debt", "location", "timeframe", "interest_rate"],
        "questions": {
            "price": "🏠 What is the estimated price of the house?",
            "savings": "💰 How much savings do you currently have?",
            "income": "💵 What is your monthly income?",
            "expenses": "📊 What are your monthly expenses?",
            "debt": "💳 Do you currently have any debts? If so, how much in total?",
            "location": "📍 Where would you like to buy the house?",
            "timeframe": "⏳ What is your target timeline (e.g., 2 years)?",
            "interest_rate": "📈 What mortgage interest rate are you expecting?",
        },
    },
    "savings": {
        "required": ["income", "expenses", "savings", "goal", "timeframe"],
        "questions": {
            "income": "💵 What is your monthly income?",
            "expenses": "📊 What are your monthly expenses?",
            "savings": "💰 How much savings do you already have?",
            "goal": "🎯 What is your savings goal?",
            "timeframe": "📅 What is your target timeframe?",
        },
    },
    "trip": {
        "required": ["destination", "dates", "budget", "preferences"],
        "questions": {
            "destination": "🌍 Where would you like to travel?",
            "dates": "📅 When do you plan to travel?",
            "budget": "💵 What is your total budget?",
            "preferences": "🎯 Any travel preferences (hotels, activities, flights)?",
        },
    },
}

def find_mortgage_rate(location):
    query = f"current average mortgage interest rate in {location or 'Europe'} {date.today().year}"
    try:
        result = DuckDuckGoSearchRun().run(query)
    except Exception:
        return None
    match = re.search(r"(\d{1,2}(?:[.,]\d{1,2})?)\s?%", result)
    return float(match.group(1).replace(",", ".")) if match else None

# --- Tool: Create Plan ---
@tool("create_plan")
def create_plan(
    plan_type: str,
    price: Optional[float] = None,
    savings: Optional[float] = None,
    income: Optional[float] = None,
    expenses: Optional[float] = None,
    debt: Optional[float] = None,
    location: Optional[str] = None,
    timeframe: Optional[str] = None,
    interest_rate: Optional[float] = None,
    accept_suggested_rate: bool = False,
    goal: Optional[str] = None,
    destination: Optional[str] = None,
    dates: Optional[str] = None,
    budget: Optional[float] = None,
    preferences: Optional[str] = None,
) -> str:
    """Build a personalized financial plan. plan_type is 'house', 'savings' or 'trip'.

    Call it every time the user gives new information for a plan, passing ONLY the
    fields the user actually stated (in any order, corrections included). Leave the others empty.
    - income and expenses are MONTHLY amounts: divide yearly figures by 12.
    - savings and debt are current totals; "no debt" means debt=0.
    - interest_rate is a percentage, e.g. 3.5 for 3.5%. If the user accepts the rate
      the tool suggested, set accept_suggested_rate=True instead.
    - goal = what the user is saving for / target amount; budget = total trip budget.
    Returns the next question to ask the user, or the full plan once the profile is complete.
    """
    session = get_session()
    plan_type = plan_type.lower().strip()
    if plan_type not in PLAN_SCHEMAS:
        return f"⚠️ I don't yet support '{plan_type}' plans (available: house, savings, trip)."

    schema = PLAN_SCHEMAS[plan_type]
    plan_state = session["plan_states"].setdefault(
        plan_type, {"answers": {}, "rate_searched": False, "suggested_rate": None}
    )
    answers = plan_state["answers"]

    if accept_suggested_rate and plan_state["suggested_rate"] is not None:
        interest_rate = plan_state["suggested_rate"]

    provided = {
        "price": price, "savings": savings, "income": income, "expenses": expenses,
        "debt": debt, "location": location, "timeframe": timeframe,
        "interest_rate": interest_rate, "goal": goal, "destination": destination,
        "dates": dates, "budget": budget, "preferences": preferences,
    }
    for field, value in provided.items():
        if value is not None and field in schema["required"]:
            answers[field] = value
            session["user_profile"][field] = value

    # Find missing
    missing = [f for f in schema["required"] if f not in answers]
    if missing:
        next_field = missing[0]

        # Auto mortgage rate (searched only once, so the flow can't loop)
        if next_field == "interest_rate" and not plan_state["rate_searched"]:
            plan_state["rate_searched"] = True
            plan_state["suggested_rate"] = find_mortgage_rate(answers.get("location"))
            if plan_state["suggested_rate"] is not None:
                return (f"I found an average mortgage rate of about {plan_state['suggested_rate']}% "
                        f"for {answers.get('location', 'your area')}. Would you like me to use this, "
                        "or do you expect a different rate?")

        return schema["questions"][next_field]

    # Completed
    session["user_profile"]["Project"] = plan_type.capitalize()
    def fmt(key, value):
        if key == "interest_rate":
            return f"{value}%"
        return f"{value:,.0f}" if isinstance(value, (int, float)) else value
    summary = "\n".join([f"- {k.replace('_', ' ').capitalize()}: {fmt(k, v)}" for k, v in answers.items()])

    if plan_type == "trip":
        health_section, level = "", None  # no income data to evaluate
    else:
        advice, level = evaluate_financial_health(answers)
        session["user_profile"]["Financial Health"] = level
        health_section = f"\n**Financial Health**\n{advice}\n"

    plan = f"""✅ Your {plan_type.capitalize()} Plan

**Profile**
{summary}
{health_section}
**Roadmap**
1. Save strategically.
2. Manage expenses.
3. Stay consistent with your timeline.
"""
    session["last_plan"] = plan

    # Suggest savings plan if needed
    if plan_type == "house":
        down_payment = 0.2 * float(answers["price"])
        if level in ["Weak", "Moderate"] or float(answers["savings"]) < down_payment:
            return (plan + f"\n👉 A 20% down payment would be {down_payment:,.0f}. "
                    "Would you like me to create a savings plan to help you reach it?")

    return plan

# --- Tool: Knowledge base (RAG from Step 2) ---
@tool("knowledge_base")
def knowledge_base(question: str) -> str:
    """Answer personal-finance concept questions (budgeting, saving, debt, credit, investing basics,
    financial goals, habits, financial freedom) from the curated YouTube course transcripts."""
    return rag_chain.invoke({"input": question, "chat_history": []})["answer"]

# --- Web Search Tool ---
@tool("web_search")
def web_search(query: str) -> str:
    """Search the web for current financial information such as rates, prices or news."""
    return DuckDuckGoSearchRun().run(query)

# --- Agent Setup ---
llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
tools = [knowledge_base, create_plan, web_search]

agent_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are 'Coach', a friendly personal-finance coach.\n"
     "- For financial concepts (budgeting, saving, debt, credit, investing basics, habits), call "
     "`knowledge_base` first and base your answer on it.\n"
     "- When the user wants a house, savings or trip plan, call `create_plan` with every detail they gave, "
     "then ask the question it returns. Ask one question at a time.\n"
     "- Use `web_search` only for current data (rates, prices, news).\n"
     "- Explain financial health like a teacher.\n"
     "- You are not a licensed financial advisor: for personal investment decisions "
     "(e.g. 'should I invest in X now?'), say you cannot give personalized investment advice, "
     "then share general educational guidance.\n"
     "- Reply in the user's language."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_openai_tools_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


**STEP 4 : EVALUATION**


In [6]:
# ===================== Cell 4: Evaluation =====================
import os
from pydantic import BaseModel, Field
from langsmith import Client, evaluate
from langchain_openai import ChatOpenAI

# ----------------- Setup -----------------
# LANGCHAIN_API_KEY is read from the .env file loaded in Cell 1
os.environ["LANGCHAIN_TRACING_V2"] = "true"
PROJECT_NAME = os.getenv("LANGCHAIN_PROJECT") or "Personal-Finance-Buddy-Evaluation"
os.environ["LANGCHAIN_PROJECT"] = PROJECT_NAME

client = Client()

# ----------------- Chatbot Wrapper -----------------
def run_chatbot_qa(example, chat_history: list) -> dict:
    """
    Wrapper for the chatbot pipeline.
    Accepts either dataset rows (with 'question' or 'input') or raw strings.
    chat_history is a list of ("human", text) / ("ai", text) tuples.
    """
    try:
        if isinstance(example, dict):
            # Handle both dataset schema ('question') and custom schema ('input')
            user_query = example.get("input") or example.get("question")
        else:
            # Raw string provided
            user_query = str(example)

        result = agent_executor.invoke({"input": user_query, "chat_history": chat_history})
        return {"answer": result["output"]}
    except Exception as e:
        return {"answer": f"⚠️ Error: {str(e)}"}

def chat_turn(user_msg, chat_history):
    resp = run_chatbot_qa(user_msg, chat_history)
    print(f"User: {user_msg}\nBot: {resp['answer']}\n")
    chat_history += [("human", user_msg), ("ai", resp["answer"])]
    return resp["answer"]

# ----------------- Test checker -----------------
# An LLM checks each answer against a criterion: it judges the meaning, not exact keywords
# ("can't" vs "cannot", "grow" vs "grows" would otherwise fail).
class Check(BaseModel):
    passed: bool = Field(description="True if the answer meets the criterion")
    reason: str = Field(description="One short sentence explaining the verdict")

checker = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(Check)
test_results = []

def check(name, conversation, criterion):
    verdict = checker.invoke(
        "You are testing a personal-finance coaching chatbot. Decide whether the chatbot meets the "
        "criterion. Judge the meaning, not the exact wording.\n\n"
        f"Conversation:\n{conversation}\n\nCriterion: {criterion}"
    )
    test_results.append((name, verdict.passed))
    print(("✅ Passed" if verdict.passed else "❌ Failed") + f" – {verdict.reason}\n")
    return verdict.passed

def check_value(name, actual, expected, tolerance=0.01):
    passed = actual is not None and abs(float(actual) - expected) <= tolerance * max(abs(expected), 1)
    test_results.append((name, passed))
    print(("✅" if passed else "❌") + f" {name}: {actual} (expected ≈ {expected})")
    return passed

def transcript(chat_history):
    return "\n".join(f"{'User' if role == 'human' else 'Coach'}: {text}" for role, text in chat_history)

# ----------------- Self-Test -----------------
print("\n--- Running quick self-test ---")

self_test_cases = [
    {"q": "What is a budget?",
     "criterion": "Explains that a budget is a plan of how income is spent and saved."},
    {"q": "What's the principle of forced saving?",
     "criterion": "Explains that money is set aside automatically or first (before spending), so saving doesn't rely on willpower."},
    {"q": "What's the difference between an emergency fund and investment savings?",
     "criterion": "Distinguishes an emergency fund (safe, accessible money for unexpected expenses) from investment savings (long-term growth, with more risk)."},
    {"q": "What's more important, paying off debt or saving?",
     "criterion": "Gives balanced guidance, e.g. prioritize high-interest debt while keeping or building an emergency fund."},
    {"q": "If I save $200 per month for 5 years at 5% annual interest, how much will I have?",
     "criterion": "Gives a final amount of roughly $13,600 (anything between $13,000 and $14,000 is correct)."},
    {"q": "Should I invest in real estate right now?",
     "criterion": "Says it cannot give personalized investment advice, and still shares general educational guidance."},
    {"q": "If I only pay the minimum on my credit card, I won’t pay interest, right?",
     "criterion": "Corrects the misconception: paying only the minimum still incurs interest on the remaining balance."},
    {"q": "What does compound interest mean in simple terms?",
     "criterion": "Explains that you earn interest on previously earned interest, so money grows faster over time."},
]

for case in self_test_cases:
    use_session(new_session())
    answer = run_chatbot_qa(case["q"], chat_history=[])["answer"]
    print(f"\nQ: {case['q']}\nA: {answer}")
    check(case["q"], f"User: {case['q']}\nCoach: {answer}", case["criterion"])

# ----------------- Multi-turn Plan Scenario -----------------
# Answers are given in a different order than the tool asks, to test field extraction.
print("\n--- Running multi-turn scenario test ---")
scenario_steps = [
    "My income is 80,000 per year",
    "I have 20,000 in savings",
    "Expenses are 2000 per month",
    "No debts",
    "House price is 250,000",
    "In Porto",
    "Within 3 years",
    "I expect a rate around 3.5%",
]

use_session(new_session())
chat_history = []
chat_turn("Help me to buy a house", chat_history)
for step in scenario_steps:
    chat_turn(step, chat_history)

# Exact checks on what the agent extracted into the profile
profile = get_session()["user_profile"]
check_value("Monthly income (80,000 / 12)", profile.get("income"), 80000 / 12)
check_value("Savings", profile.get("savings"), 20000)
check_value("Monthly expenses", profile.get("expenses"), 2000)
check_value("Debt", profile.get("debt"), 0)
check_value("House price", profile.get("price"), 250000)
check_value("Interest rate", profile.get("interest_rate"), 3.5)
check("House plan scenario", transcript(chat_history[-2:]),
      "The coach presents a completed house plan with the user's profile (price, savings, income, "
      "expenses, debt, location) and a financial health evaluation.")

# ----------------- Complex Multi-turn Scenario -----------------
print("\n--- Running complex multi-turn scenario test ---")
complex_steps = [
    "I'm not sure if I should save more or pay off my credit card debt first. I owe about $5,000.",
    "I also have some old student loans, but I only make a small payment each month, so I'm not worried about them.",
    "My salary is $60,000, but last month I told you it was $50,000. Sorry, I was just guessing before.",
]

use_session(new_session())
chat_history = []
chat_turn("I'm worried about my financial future. What should I do?", chat_history)
for step in complex_steps:
    chat_turn(step, chat_history)

check("Complex scenario", transcript(chat_history),
      "Over the conversation the coach (1) gives guidance on paying off the credit card debt versus saving, "
      "(2) does not ignore the student loans, and (3) uses the corrected salary of $60,000, not $50,000.")

passed = sum(ok for _, ok in test_results)
print(f"\n📊 Self-tests: {passed}/{len(test_results)} passed")
for name, ok in test_results:
    if not ok:
        print(f"   ❌ {name}")

# ----------------- LLM-as-judge Evaluation -----------------
class Grade(BaseModel):
    correctness: int = Field(description="1-5: does the answer agree with the reference answer?")
    clarity: int = Field(description="1-5: is the answer clear and easy to understand?")
    helpfulness: int = Field(description="1-5: is the answer useful and actionable for a beginner?")
    reasoning: str = Field(description="One or two sentences justifying the scores.")

judge = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(Grade)

def target(inputs: dict) -> dict:
    use_session(new_session())
    return run_chatbot_qa(inputs, chat_history=[])

def coach_judge(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    question = inputs.get("input") or inputs.get("question")
    reference = (reference_outputs or {}).get("answer") or (reference_outputs or {}).get("output") \
        or next(iter((reference_outputs or {}).values()), "")
    grade = judge.invoke(
        "You are grading a personal-finance coaching chatbot.\n\n"
        f"Question: {question}\n\nReference answer: {reference}\n\n"
        f"Chatbot answer: {outputs.get('answer')}"
    )
    return {"results": [
        {"key": key, "score": getattr(grade, key) / 5, "comment": grade.reasoning}
        for key in ("correctness", "clarity", "helpfulness")
    ]}

# ----------------- Run Evaluation -----------------
DATASET_NAME = "Financial-Coach-Complex-Tests"
try:
    dataset_found = client.has_dataset(dataset_name=DATASET_NAME)
except Exception as e:  # missing / invalid LangSmith key
    print(f"⚠️ Could not reach LangSmith ({e.__class__.__name__}): check LANGCHAIN_API_KEY in .env")
    dataset_found = False

if dataset_found:
    print("\n--- Running full evaluation on dataset ---")
    results = evaluate(
        target,
        data=DATASET_NAME,
        evaluators=[coach_judge],
        experiment_prefix="finance-coach",
        client=client,
    )
    print("\n✅ Evaluation complete! Check LangSmith dashboard.")
else:
    print(f"ℹ️ LangSmith dataset '{DATASET_NAME}' not found: skipping the dataset evaluation.")



--- Running quick self-test ---


> Entering new AgentExecutor chain...

Invoking: `knowledge_base` with `{'question': 'What is a budget?'}`


A budget is a plan that helps you manage your money. It shows you how much money you have, how much money you need to spend on things, and how much money you can save or use for other goals. It helps you make smart decisions with your money and avoid problems like overspending, debt, or running out of money. Essentially, a budget is a plan for the money that's going to be coming in and what you're going to be doing with it.Un budget es un plan que te ayuda a gestionar tu dinero. Te muestra cuánto dinero tienes, cuánto necesitas gastar en cosas y cuánto puedes ahorrar o usar para otros objetivos. Te ayuda a tomar decisiones inteligentes con tu dinero y a evitar problemas como gastar de más, endeudarte o quedarte sin dinero. Básicamente, un presupuesto es un plan para el dinero que va a entrar y lo que vas a hacer con él.

> Finished chain.

Q: W

/Users/katyd/Downloads/personal-finance-buddy-main/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Check(passed=True, reason...g income and expenses.'), input_type=Check])
  return self.__pydantic_serializer__.to_python(


✅ Passed – The explanation clearly defines a budget as a plan for managing income and expenses.



> Entering new AgentExecutor chain...

Invoking: `knowledge_base` with `{'question': 'What is the principle of forced saving?'}`


My sources don't cover the principle of forced saving.The principle of forced saving involves setting up a system where you automatically save a portion of your income before you have the chance to spend it. This can be done through mechanisms like automatic transfers to a savings account, payroll deductions, or investing in retirement accounts. The idea is to make saving a non-negotiable part of your financial routine, ensuring that you consistently set aside money for future needs or goals without relying on willpower alone. This method helps in building savings over time and can prevent overspending.

> Finished chain.

Q: What's the principle of forced saving?
A: The principle of forced saving involves setting up a system where you automatically save a por

/Users/katyd/Downloads/personal-finance-buddy-main/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Check(passed=True, reason...tic saving mechanisms.'), input_type=Check])
  return self.__pydantic_serializer__.to_python(


✅ Passed – The explanation clearly describes automatic saving mechanisms.



> Entering new AgentExecutor chain...

Invoking: `knowledge_base` with `{'question': 'difference between an emergency fund and investment savings'}`


My sources don't cover the specific differences between an emergency fund and investment savings.An emergency fund and investment savings serve different purposes in personal finance:

1. **Emergency Fund**:
   - **Purpose**: It's a safety net for unexpected expenses such as medical emergencies, car repairs, or sudden job loss.
   - **Accessibility**: Should be easily accessible, typically kept in a savings account or money market account.
   - **Amount**: Generally recommended to cover 3-6 months of living expenses.
   - **Risk**: Low risk, as the primary goal is to preserve capital and ensure liquidity.

2. **Investment Savings**:
   - **Purpose**: Intended for long-term growth, such as retirement or wealth building.
   - **Accessibility**: Not immediately acc

...



> Entering new AgentExecutor chain...

Invoking: `knowledge_base` with `{'question': 'Should I invest or pay off high-interest debt first?'}`


My sources suggest that you should prioritize paying off high-interest debt first. Rank your debt by interest rate from highest to lowest and focus on paying off any debt with an interest rate over 8% as soon as possible. This approach is considered the fastest way to improve your financial situation.It's generally recommended to prioritize paying off high-interest debt, like your credit card debt at 20% interest, before investing. This is because the interest on your debt is likely higher than the returns you would earn from most investments. By focusing on paying off this debt, you can improve your financial situation more effectively.

> Finished chain.


/Users/katyd/Downloads/personal-finance-buddy-main/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Grade(correctness=5, clar...useful for a beginner."), input_type=Grade])
  return self.__pydantic_serializer__.to_python(
4it [00:22,  5.25s/it]



> Entering new AgentExecutor chain...

Invoking: `knowledge_base` with `{'question': 'how much should I put into an emergency fund'}`


My sources suggest that you should aim to save three to six months' worth of your fundamental expenses in an emergency fund. If you have a stable job, start with three months. If your income is unpredictable, aim for six months. For example, if your monthly expenses are $1,500, you should have at least $4,500 saved.To build an emergency fund, it's recommended to save three to six months' worth of your essential expenses. Since your monthly expenses are €2,000, you should aim to have between €6,000 and €12,000 in your emergency fund. If your job is stable, you might start with the lower end of that range, but if your income is unpredictable, aim for the higher end.

> Finished chain.


/Users/katyd/Downloads/personal-finance-buddy-main/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Grade(correctness=5, clar...ulness for a beginner."), input_type=Grade])
  return self.__pydantic_serializer__.to_python(
5it [00:30,  6.08s/it]


✅ Evaluation complete! Check LangSmith dashboard.


**GRADIO**

In [ ]:
# ----------------- Cell 5: Gradio UI -----------------
import os
import tempfile

import gradio as gr
import pandas as pd
import speech_recognition as sr
from fpdf import FPDF

# ---------- Utilities ----------
def speech_to_text(audio_file):
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_file) as source:
        audio = recognizer.record(source)
    try:
        return recognizer.recognize_google(audio)
    except Exception:
        return "⚠️ Speech recognition error."

def message_text(content):
    # Gradio 6 returns message content as a list of parts: [{"type": "text", "text": "..."}]
    if isinstance(content, list):
        return " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    return str(content)

def temp_path(suffix):
    # Unique file per export so two users never overwrite each other's file
    fd, path = tempfile.mkstemp(suffix=suffix)
    os.close(fd)
    return path

def pdf_safe(text):
    # FPDF built-in fonts are latin-1 only: drop emojis and markdown bold markers
    text = str(text).replace("**", "")
    for src, dst in {"’": "'", "‘": "'", "“": '"', "”": '"', "–": "-", "—": "-", "≥": ">=", "≤": "<=", "→": "->"}.items():
        text = text.replace(src, dst)
    return text.encode("latin-1", "ignore").decode("latin-1").strip()

def save_conversation_as_pdf(history, session):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "Conversation Export", new_x="LMARGIN", new_y="NEXT", align="C")
    pdf.ln(5)

    pdf.set_font("Helvetica", "B", 12)
    pdf.cell(0, 8, "User Profile", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", size=11)
    for k, v in session["user_profile"].items():
        pdf.multi_cell(0, 7, pdf_safe(f"- {k}: {v}"), new_x="LMARGIN", new_y="NEXT")

    pdf.ln(5)
    pdf.set_font("Helvetica", "B", 12)
    pdf.cell(0, 8, "Conversation", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", size=11)
    for msg in history:
        pdf.multi_cell(0, 7, pdf_safe(f"{msg['role']}: {message_text(msg['content'])}"), new_x="LMARGIN", new_y="NEXT")
        pdf.ln(2)

    path = temp_path(".pdf")
    pdf.output(path)
    return path

def save_report_as_excel(session):
    profile = session["user_profile"]
    profile_df = pd.DataFrame(list(profile.items()), columns=["Profile Key", "Value"])

    if "income" in profile:
        advice, level = evaluate_financial_health(profile)
        health_rows = [(k, round(v, 2)) for k, v in financial_metrics(profile).items()]
        health_rows += [("Level", level), ("Advice", advice)]
    else:
        health_rows = [("Level", "Not evaluated yet (no income data)")]
    health_df = pd.DataFrame(health_rows, columns=["Metric", "Evaluation"])

    plan_text = session["last_plan"] or "No plan generated yet."
    plan_df = pd.DataFrame({"Plan": [line.replace("**", "") for line in plan_text.splitlines() if line.strip()]})

    path = temp_path(".xlsx")
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        profile_df.to_excel(writer, sheet_name="User Profile", index=False)
        health_df.to_excel(writer, sheet_name="Financial Health", index=False)
        plan_df.to_excel(writer, sheet_name="Action Plan", index=False)
    return path

# ---------- Chatbot ----------
def chatbot_interface(message, audio, history, session):
    use_session(session)
    if audio:
        message = speech_to_text(audio)
    if not message or not message.strip():
        return history, session["user_profile"], session, "", None

    chat_history = [("human" if m["role"] == "user" else "ai", message_text(m["content"])) for m in history]
    result = agent_executor.invoke({"input": message, "chat_history": chat_history})

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": result["output"]},
    ]
    # Clear the textbox and the audio input after sending
    return history, session["user_profile"], session, "", None

def reset_profile():
    return [], {}, new_session()

# ---------- Gradio UI ----------
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 Personal Finance & Planning Assistant")
    session_state = gr.State(new_session())  # one session per browser visitor

    with gr.Row():
        msg = gr.Textbox(label="💬 Your message")
        audio = gr.Audio(sources=["microphone"], type="filepath", label="🎤 Speak")

    with gr.Row():
        send_btn = gr.Button("Send")
        reset_btn = gr.Button("🔄 Reset Profile")
        save_pdf_btn = gr.Button("💾 Save Conversation as PDF")
        save_excel_btn = gr.Button("📊 Save Report as Excel")

    chatbot = gr.Chatbot(label="Conversation")
    profile_display = gr.JSON(label="📂 Current User Profile")
    pdf_output = gr.File(label="📄 Download PDF")
    excel_output = gr.File(label="📊 Download Excel")

    # Button actions
    chat_inputs = [msg, audio, chatbot, session_state]
    chat_outputs = [chatbot, profile_display, session_state, msg, audio]
    send_btn.click(chatbot_interface, chat_inputs, chat_outputs)
    msg.submit(chatbot_interface, chat_inputs, chat_outputs)
    reset_btn.click(reset_profile, None, [chatbot, profile_display, session_state])
    save_pdf_btn.click(save_conversation_as_pdf, [chatbot, session_state], pdf_output)
    save_excel_btn.click(save_report_as_excel, session_state, excel_output)

# Set GRADIO_SHARE=1 in .env to get a temporary public link
demo.launch(share=os.getenv("GRADIO_SHARE") == "1")
